In [ ]:
## install/import dependancies ##

import torch
import transformers

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

## working well on colab

In [ ]:
## import libraries ##

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel, AutoTokenizer

#working on colab

In [ ]:
## defining a model ##

class SBERTModel(nn.Module):

    def __init__(self, model_name="bert-base-uncased"):
        super(SBERTModel, self).__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):

        token_embeddings = model_output.last_hidden_state

        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sentence_embedding = self.mean_pooling(outputs, attention_mask)

        return sentence_embedding


## Runs well on colab and doesnot have any infinite loops until now

In [ ]:
##first test case##

sentences = [
    "SSH login failed from 10.0.0.5",
    "Multiple authentication failures detected"
]

inputs = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) for k, v in inputs.items()}

embeddings = model(
    inputs["input_ids"],
    inputs["attention_mask"]
)

print(embeddings.shape)

##passed the first test case##

In [ ]:
##define cosine similarity

def cosine_similarity(a, b):
    return F.cosine_similarity(a, b, dim=0)

##works with a simple fix

In [ ]:
## test cosine similarity##
sim = cosine_similarity(embeddings[1], embeddings[1])
print(sim)

In [ ]:
##loss function training##
## calculate a loss function with this in the form of [0,1]

# import torch
# import torch.nn as nn
# import torch.nn.functional as F

class CosineSimilarityLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.MSELoss()

    def forward(self, emb1, emb2, labels):
        cos_sim = F.cosine_similarity(emb1, emb2)

        # normalize from [-1,1] → [0,1]
        cos_sim = (cos_sim + 1) / 2

        labels = labels.float()

        return self.loss(cos_sim, labels)



